# Phân loại bất thường tim mạch từ ECG 12 chuyển đạo (PTB-XL) — notebook chạy trên Google Colab

Notebook này tái lập kết quả chính của nhóm: dữ liệu → mô hình cơ sở → so sánh CNN / GRU / LSTM / attention → ablation → tổng hợp.
Mã nguồn thật nằm trong thư mục `ptbxl_ecg/`; notebook chỉ gọi các lệnh.

**Cách dùng**
1. `Runtime` → `Change runtime type` → chọn GPU (T4).
2. Điền ô **Cấu hình** (đường dẫn repo, thành viên đang chạy).
3. Chạy lần lượt từ trên xuống. Mỗi phần có nhãn **[A]**, **[B]**, **[C]**: mỗi thành viên chỉ chạy phần của mình.
4. Kết quả và nhật ký được ghi vào Google Drive nên không mất khi Colab ngắt. Chạy lại ô huấn luyện sẽ bỏ qua các lượt đã xong.

> Notebook chỉ chạy trọn vẹn khi các phần TODO trong `ptbxl_ecg/` đã được cài đặt (kiểm tra bằng ô chạy test bên dưới).

## 0. Cấu hình

In [ ]:
# ====== CẦN SỬA ======
REPO_URL = "https://github.com/TiepKun/ptbxl-ecg.git"   # repo công khai của nhóm
BRANCH = "main"
MEMBER = "A"          # "A" (Hiếu), "B" (Tiếp) hoặc "C" (Hùng): dùng để đặt tên file log riêng

# Việc của từng người (chỉnh theo vai trò, xem docs/PLAN.md)
RUN_BASELINE = (MEMBER == "A")                      # [A] baseline LR/RF
MODELS_TO_RUN = {"A": [], "B": ["cnn_simple", "resnet1d", "cnn_attn"], "C": ["gru", "lstm"]}[MEMBER]
RUN_ABLATION = (MEMBER == "C")                      # [C] ablation số chuyển đạo + augmentation
BEST_MODEL = None     # None: tự chọn theo val AUC trung bình trong các file log đã có; hoặc gõ tên, ví dụ "resnet1d"

SEEDS = [0, 1, 2]     # yêu cầu: ít nhất 3 seed cho mỗi cấu hình
EPOCHS = 30
BATCH_SIZE = 128
# =====================


## 1. Môi trường: GPU, Google Drive, mã nguồn

In [ ]:
!nvidia-smi -L
import torch
print("PyTorch", torch.__version__, "| GPU:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Chưa bật GPU: Runtime > Change runtime type > T4 GPU"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
DRIVE = "/content/drive/MyDrive/ptbxl-ecg"          # dữ liệu, runs và log được lưu ở đây
RAW, PROC, RUNS = f"{DRIVE}/raw", f"{DRIVE}/processed", f"{DRIVE}/runs"
LOG_DIR = f"{DRIVE}/logs"
LOG_CSV = f"{LOG_DIR}/experiments_{MEMBER}.csv"
for d in (RAW, PROC, RUNS, LOG_DIR):
    os.makedirs(d, exist_ok=True)
print("Thư mục Drive:", DRIVE)

In [ ]:
import os
if not os.path.exists("/content/ptbxl-ecg"):
    !git clone -b {BRANCH} {REPO_URL} /content/ptbxl-ecg
%cd /content/ptbxl-ecg
!git pull --ff-only
# Colab đã có sẵn torch; chỉ cài các gói còn lại
!grep -v "^torch" requirements.txt > /tmp/req.txt && pip install -q -r /tmp/req.txt
!git log -1 --oneline

## 2. Kiểm tra mã trước khi chạy dữ liệu thật

Mỗi người chạy bộ test của mình, cả nhóm chạy `test_pipeline.py` cuối cùng. Test dùng dữ liệu giả lập nên chạy trong vài chục giây.
Nếu còn test đỏ với `NotImplementedError` nghĩa là phần TODO tương ứng chưa được cài đặt.

In [ ]:
!python -m pytest -q tests/test_A_data_eval.py tests/test_B_cnn_train.py tests/test_C_rnn_ablation.py tests/test_pipeline.py

## 3. Dữ liệu PTB-XL  **[A]**

In [ ]:
# Tải PTB-XL v1.0.3 (bản 100 Hz) vào Drive. Có thể chạy lại để tiếp tục nếu bị ngắt.
# Lưu ý: script này chưa được thử trên PhysioNet thật. Nếu lỗi, tải thủ công từ
# https://physionet.org/content/ptb-xl/1.0.3/ rồi đặt ptbxl_database.csv, scp_statements.csv, records100/ vào thư mục RAW.
if not os.path.exists(f"{RAW}/ptbxl_database.csv"):
    !bash scripts/download_data.sh {RAW}
else:
    print("Đã có dữ liệu thô tại", RAW)

In [ ]:
# Đọc tín hiệu, ánh xạ nhãn sang 5 siêu lớp, lưu X.npy / Y.npy / folds.npy
if not os.path.exists(f"{PROC}/X.npy"):
    !python -m ptbxl_ecg.prepare_data --raw-dir {RAW} --out-dir {PROC}
else:
    print("Đã có dữ liệu đã xử lý tại", PROC)

In [ ]:
# Thống kê mô tả: số bản ghi, phân bố nhãn theo fold (ghi vào results/)
!python -m ptbxl_ecg.eda --data-dir {PROC} --out-dir results

## 4. Mô hình cơ sở: đặc trưng thủ công + LR / RF  **[A]**

In [ ]:
if RUN_BASELINE:
    !python -m ptbxl_ecg.baseline --data-dir {PROC} --log-csv {LOG_CSV} --seeds {" ".join(map(str, SEEDS))}
else:
    print("Bỏ qua (không phải phần của", MEMBER, ")")

## 5. So sánh kiến trúc (nhóm `main`)  **[B] [C]**

Mỗi người chỉ chạy các mô hình của mình (`MODELS_TO_RUN`). Chạy lại sau khi Colab ngắt sẽ tự bỏ qua các lượt đã xong.

In [ ]:
if MODELS_TO_RUN:
    models = " ".join(MODELS_TO_RUN)
    !python -m ptbxl_ecg.ablation --group main --models {models} --seeds {" ".join(map(str, SEEDS))} --dry-run
else:
    print("Bỏ qua (không phải phần của", MEMBER, ")")

In [ ]:
if MODELS_TO_RUN:
    !python -m ptbxl_ecg.ablation --group main --models {models} --seeds {" ".join(map(str, SEEDS))} \
        --epochs {EPOCHS} --batch-size {BATCH_SIZE} --data-dir {PROC} --out-dir {RUNS} --log-csv {LOG_CSV}

## 6. Ablation: số chuyển đạo và augmentation  **[C]**

Ablation chạy trên **một** mô hình, chọn theo **val AUC** (không dùng test để chọn). Cần log của cả nhóm ở nhóm `main`:
mỗi người `git add logs/experiments_<A|B|C>.csv` (chép từ Drive ở phần cuối), `git push`, rồi người chạy ablation `git pull`
và chép các file log vào `logs/` trước khi chạy ô dưới. Mô hình `cnn_simple` (baseline học sâu) không được chọn.

In [ ]:
import glob, pandas as pd
if RUN_ABLATION:
    logs = glob.glob("logs/experiments_*.csv") + glob.glob(f"{LOG_DIR}/experiments_*.csv")
    df = pd.concat([pd.read_csv(f) for f in logs], ignore_index=True)
    main_runs = df[(df["group"] == "main") & (df["model"] != "cnn_simple")]
    ranking = main_runs.groupby("model")["val_auc"].agg(["mean", "count"]).sort_values("mean", ascending=False)
    display(ranking)
    if BEST_MODEL is None:
        BEST_MODEL = ranking.index[0]
    print("Mô hình dùng cho ablation:", BEST_MODEL)

In [ ]:
if RUN_ABLATION:
    seeds = " ".join(map(str, SEEDS))
    !python -m ptbxl_ecg.ablation --group leads --model {BEST_MODEL} --seeds {seeds} --dry-run
    !python -m ptbxl_ecg.ablation --group leads --model {BEST_MODEL} --seeds {seeds} \
        --epochs {EPOCHS} --batch-size {BATCH_SIZE} --data-dir {PROC} --out-dir {RUNS} --log-csv {LOG_CSV}
    !python -m ptbxl_ecg.ablation --group aug --model {BEST_MODEL} --seeds {seeds} \
        --epochs {EPOCHS} --batch-size {BATCH_SIZE} --data-dir {PROC} --out-dir {RUNS} --log-csv {LOG_CSV}
else:
    print("Bỏ qua (không phải phần của", MEMBER, ")")

## 7. Tổng hợp kết quả và phân tích  **[A] [C]**

In [ ]:
# Gộp mọi file log của nhóm thành logs/experiments.csv (log nộp kèm) rồi tổng hợp mean ± std theo seed
import glob, os, pandas as pd
os.makedirs("logs", exist_ok=True)
files = sorted(set(glob.glob("logs/experiments_*.csv") + glob.glob(f"{LOG_DIR}/experiments_*.csv")))
print("Gộp:", files)
merged = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
merged = merged.drop_duplicates("run_name", keep="last")   # cùng một lượt có thể nằm ở cả Drive và repo
merged.to_csv("logs/experiments.csv", index=False)
print(len(merged), "lượt chạy")
!python -m ptbxl_ecg.aggregate --log-csv logs/experiments.csv --out-dir results

In [ ]:
from IPython.display import Image, Markdown, display
display(Markdown(open("results/summary.md", encoding="utf-8").read()))
display(Image("results/test_auc.png"))

In [ ]:
# [C] Độ quan trọng theo chuyển đạo (saliency) của mô hình tốt nhất, seed 0
if RUN_ABLATION:
    run_dir = f"{RUNS}/main_{BEST_MODEL}_12_aug1_s0"
    !python -m ptbxl_ecg.saliency --run-dir {run_dir} --data-dir {PROC} --out-dir results
    from glob import glob
    for f in glob("results/saliency_*.png"):
        display(Image(f))

## 8. Lưu kết quả

Chép log và kết quả của bạn ra Drive và tải về để đưa vào repo (`git add logs/ results/`).
Nhớ rằng nhật ký thí nghiệm là bằng chứng chấm điểm: số liệu trong báo cáo phải khớp với `logs/experiments.csv`.

In [ ]:
import shutil, os
os.makedirs("logs", exist_ok=True)
shutil.copy(LOG_CSV, f"logs/experiments_{MEMBER}.csv") if os.path.exists(LOG_CSV) else None
shutil.make_archive(f"{DRIVE}/ket_qua_{MEMBER}", "zip", ".", "results")
shutil.copytree("results", f"{DRIVE}/results", dirs_exist_ok=True)
from google.colab import files
files.download(f"logs/experiments_{MEMBER}.csv") if os.path.exists(f"logs/experiments_{MEMBER}.csv") else print("Chưa có log")